# 9.15 confidence 是「有幾成機會答對」嗎？


三個候選機率是0.6、0.2、0.2。依[9.9](https://birdhackor.github.io/tiny-perceptron-vlm/9.9.html)的定義，信心就是最大值0.6，也就是[9.10](https://birdhackor.github.io/tiny-perceptron-vlm/9.10.html)拿去和正確率比較、需要校準的那個量。官方 Jev 的 Choice 也回傳一個叫 confidence 的欄位，卻不是直接複製0.6；本課把它叫做**集中度（concentration）**，避免和信心混用。這個差別會影響服務門檻，所以在把欄位接進程式前先算一次。

截至2026-10-20的官方文件，對 n 個 Choice 候選，使用

\[
c=\frac{p_{\max}-1/n}{1-1/n},\qquad n>1。
\]

`p_max` 是最大候選機率；`1/n` 是均勻分布的最大值。分子先減掉均勻基準，分母再把完全集中時的值縮放到1。因此均勻分布的 c 是0，單一候選拿到全部機率時是1。它描述**分布相對均勻基準有多集中**，不是由真實答對紀錄算出的頻率。

下面的變數沿用官方欄位名 `confidence`，但它存的是集中度，不是9.9程式裡那個最大機率；印出時標成「集中度」。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
p = [0.6, 0.2, 0.2]
n = len(p)
confidence = (max(p) - 1 / n) / (1 - 1 / n)
print("最大機率", max(p), "集中度", round(confidence, 6))

結果是0.6與0.4，算式是(0.6−1/3)/(1−1/3)=0.2667/0.6667=0.4。不能把集中度0.4解讀成這題只有四成機會對，也不能把0.6當成已校準的正確率。模型若把錯誤候選報為0.99，集中度仍高；要知道這類題實際多常答對，仍得用[9.14](https://birdhackor.github.io/tiny-perceptron-vlm/9.14.html)的測試資料逐題對照正確答案。

候選數也會影響公式。同樣最大值0.6，在二選一時 c 是0.2，在三選一時是0.4；兩個問題的候選空間不同，不能只看 c 就說後者更可靠。只有一個候選時公式分母為0，也失去比較含義，應由介面拒絕或另訂行為，不能直接套用。

Noul 的答案本身是 yes 機率，官方不另回傳一個 confidence。Score 有次序，官方另有與等級分布有關的摘要方式。第19章機率判斷器的選項、資訊充分與有限等級三種問題，都回傳完整候選機率，並把上面的最大候選公式另標為 `concentration`。這個摘要沒有實作官方 Score 的 confidence，也不能直接當成已在測試資料上驗過的答對機率。

這也解釋一個容易混淆的情形：當「需要澄清」本來就是正確答案，模型可以對它非常集中。低集中度不是缺資訊的唯一表達；缺資訊已在[9.13](https://birdhackor.github.io/tiny-perceptron-vlm/9.13.html)做成可監督的問題。

練習：把程式第一行改成 `p = [0.2, 0.6, 0.2]`，先預測印出的兩個數字都不變，再執行核對。接著在最後加一行 `print(p.index(max(p)))`，它找出最大值在清單裡的位置：原本是0，現在是1。集中度不變，選中的候選意義卻改了；任何只回一個 confidence 的報表，都不足以說明模型到底判了什麼。

來源：[TypeSafe confidence 的 Choice／Noul 定義](https://docs.typesafe.ai/confidence)。
